# 🎬 HỆ THỐNG LỒNG TIẾNG PHIM HỒNG QUẢ (HONGGUO DUBBING) TRÊN GOOGLE COLAB

Notebook này thực hiện quy trình tự động tải, ghép video, nhận diện giọng nói (ASR), dịch thuật AI và lồng tiếng tự động (VieNeu-TTS) cho phim ngắn Hồng Quả bằng script `run_hongguo_pipeline.py` **hoàn toàn trực tiếp (Direct In-Process)**:
- ❌ **Không cần chạy server Backend** (tiết kiệm tối đa RAM và CPU).
- ❌ **Không cần Cloudflare Tunnel** hay LocalTunnel.
- ⚡ Toàn bộ tác vụ chạy mượt mà ngay trong cell terminal của Colab.

> 💡 **Khuyến nghị**: Bật GPU trong **Runtime > Change runtime type > T4 GPU** để VieNeu-TTS tổng hợp giọng đọc nhanh nhất.

In [ ]:
#@title 📦 Bước 1: Cài đặt Môi trường (Clone Repo, Cài Dependencies & Đăng Ký Thiết Bị)
#@markdown Nhấn nút **Run** để tự động cài đặt toàn bộ môi trường (chỉ cần chạy 1 lần):
#@markdown 1. Clone/Cập nhật mã nguồn từ GitHub
#@markdown 2. Cài đặt FFmpeg, Node.js và các thư viện Python/Node
#@markdown 3. Tự động đăng ký thiết bị lấy Device ID & Install ID cho Hồng Quả

import os
import sys
import json
from pathlib import Path

print("🚀 [1/3] Chuẩn bị mã nguồn dự án...")
os.chdir("/content")
if not os.path.exists("/content/dubbing"):
    !git clone https://github.com/kinyias/dubbing.git /content/dubbing
else:
    %cd /content/dubbing
    !git pull
%cd /content/dubbing

print("\n📦 [2/3] Cài đặt System Dependencies (FFmpeg, Node.js)...")
!apt-get update -qq
!apt-get install -y ffmpeg nodejs npm -qq

print("\n🐍 Cài đặt Node.js & Python dependencies...")
!cd /content/dubbing/backend && npm install
!pip install -r /content/dubbing/requirements.txt -q

print("\n📲 [3/3] Đăng ký thiết bị & Khởi tạo cấu hình ban đầu...")
backend_path = os.path.abspath("/content/dubbing/backend")
if backend_path not in sys.path:
    sys.path.insert(0, backend_path)
sys.path.insert(0, os.path.join(backend_path, "service", "liushen"))

# 1. Đăng ký thiết bị động
device_id = ""
install_id = ""
try:
    from backend.service.liushen.device_register import device_register
    reg_res = device_register()
    device_id = reg_res.get("device_id", "")
    install_id = reg_res.get("install_id", "")
    print(f"✅ Đăng ký thiết bị thành công! Device ID: {device_id} | Install ID: {install_id}")
except Exception as e:
    print(f"⚠️ Cảnh báo đăng ký thiết bị: {e}")

# 2. Tạo file .env
env_content = f"""DUANJU_DEVICE_ID={device_id}
DUANJU_INSTALL_ID={install_id}
DUANJU_PLATFORM=android
APP_PORT=8000
OPEN_BROWSER=0
FLASK_DEBUG=0
FFMPEG_BIN=ffmpeg
"""
with open("/content/dubbing/.env", "w", encoding="utf-8") as f:
    f.write(env_content)

# 3. Tạo file settings.json từ settings.example.json nếu chưa có
settings_example_path = "/content/dubbing/backend/settings.example.json"
settings_path = "/content/dubbing/backend/settings.json"
if os.path.exists(settings_example_path) and not os.path.exists(settings_path):
    with open(settings_example_path, "r", encoding="utf-8") as f:
        settings = json.load(f)
    settings["ffmpegPath"] = "ffmpeg"
    with open(settings_path, "w", encoding="utf-8") as f:
        json.dump(settings, f, indent=4, ensure_ascii=False)

print("🎉 Môi trường đã sẵn sàng! Không cần chạy server backend, bạn có thể chuyển sang Bước 2.")

In [ ]:
#@title ⚙️ Bước 2: Thiết lập Cấu hình AI (API Key, Model, CapCut TDID)
#@markdown Cập nhật các thông số AI dịch thuật và CapCut ASR trực tiếp vào settings.json:

CUSTOM_API_ENDPOINT = "" #@param {type:"string"}
CUSTOM_API_KEY = "" #@param {type:"string"}
CUSTOM_MODEL = "" #@param {type:"string"}
CAPCUT_TDID = "" #@param {type:"string"}
SHOW_SETTINGS = True #@param {type:"boolean"}

%cd /content/dubbing
cmd = [sys.executable, "run_hongguo_pipeline.py", "--only-settings"]
if CUSTOM_API_ENDPOINT.strip():
    cmd.extend(["--custom-endpoint", CUSTOM_API_ENDPOINT.strip()])
if CUSTOM_API_KEY.strip():
    cmd.extend(["--custom-key", CUSTOM_API_KEY.strip()])
if CUSTOM_MODEL.strip():
    cmd.extend(["--custom-model", CUSTOM_MODEL.strip()])
if CAPCUT_TDID.strip():
    cmd.extend(["--capcut-tdid", CAPCUT_TDID.strip()])

!{" ".join(cmd)}

if SHOW_SETTINGS:
    !python run_hongguo_pipeline.py --show-settings

In [ ]:
#@title 🔍 Bước 3 (Tùy chọn): Xem Chi Tiết Phim & Danh Sách Tập
#@markdown Dán **series_id** hoặc URL link phim vào đây để xem trước tiêu đề, số lượng tập và thời lượng:

SERIES_ID_OR_URL = "74123456789" #@param {type:"string"}

%cd /content/dubbing
!python run_hongguo_pipeline.py --series-id "{SERIES_ID_OR_URL.strip()}" --detail-only

In [ ]:
#@title 🎬 Bước 4: Chạy Lồng Tiếng Phim Hồng Quả (Chạy Trực Tiếp Không Cần Server)
#@markdown Thiết lập các tùy chọn và nhấn nút **Run** để bắt đầu quy trình lồng tiếng tự động:

SERIES_ID = "74123456789" #@param {type:"string"}
EPISODE_RANGE = "1-3" #@param {type:"string"}
VOICE_ID = "Ng\u1ecdc Huy\u1ec1n" #@param ["Ng\u1ecdc Huy\u1ec1n", "V\u0103n Khoa", "Qu\u1ef3nh Trang", "Ho\u00e0i B\u1ea3o", "Th\u1ea3o Vy", "Minh Th\u01b0", "Tr\u00fac Ly", "B\u1ea3o Tr\u1ea7n"]
TTS_SPEED = 1.0 #@param {type:"slider", min:0.5, max:2.0, step:0.05}
TRANSCRIBE_ENGINE = "capcut" #@param ["capcut", "bcut", "groq", "auto"]
FIT_MODE = "natural_flow" #@param ["natural_flow", "speed_up_tts", "stretch_video", "speed_voice", "fixed"]
MAX_DOWNLOAD_WORKERS = 3 #@param {type:"integer"}
PARALLEL_TRANSLATE_JOBS = 6 #@param {type:"integer"}
CUSTOM_TITLE = "" #@param {type:"string"}

%cd /content/dubbing
cmd = [
    "python", "run_hongguo_pipeline.py",
    "--series-id", f'"{SERIES_ID.strip()}"',
    "--range", f'"{EPISODE_RANGE.strip()}"',
    "--voice", f'"{VOICE_ID}"',
    "--tts-speed", str(TTS_SPEED),
    "--transcribe-engine", TRANSCRIBE_ENGINE,
    "--fit-mode", FIT_MODE,
    "--max-workers", str(MAX_DOWNLOAD_WORKERS),
    "--parallel-jobs", str(PARALLEL_TRANSLATE_JOBS),
]

if CUSTOM_TITLE.strip():
    cmd.extend(["--title", f'"{CUSTOM_TITLE.strip()}"'])

!{" ".join(cmd)}

In [ ]:
#@title 💾 Bước 5 (Tùy chọn): Xem & Tải Video Thành Phẩm về máy / Google Drive
import os
import glob

video_files = glob.glob("/content/dubbing/downloads/**/*_dubbed.mp4", recursive=True) + glob.glob("/content/dubbing/**/*dubbed*.mp4", recursive=True)
video_files = sorted(list(set(video_files)))

if video_files:
    print(f"🎬 Tìm thấy {len(video_files)} video đã lồng tiếng:")
    for idx, v in enumerate(video_files):
        size_mb = os.path.getsize(v) / (1024 * 1024)
        print(f" [{idx + 1}] {v} ({size_mb:.2f} MB)")
    
    # Bỏ comment 2 dòng dưới để tự động tải video mới nhất về máy qua trình duyệt:
    # from google.colab import files
    # files.download(video_files[-1])
else:
    print("ℹ️ Chưa có file video lồng tiếng nào trong thư mục downloads.")

# Sao chép sang Google Drive (bỏ comment 3 dòng dưới nếu muốn lưu vĩnh viễn trên Drive):
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r /content/dubbing/downloads /content/drive/MyDrive/HongGuo_Dubbing/